In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
%sql
use catalog projectv1

In [0]:
%sql
use schema projectschema

### **Data Reading From Source**

In [0]:
df = spark.sql("select * from tblCustomersSilver")

**Removing Duplicates**

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

In [0]:
df.limit(10).display()

In [0]:
%sql
describe tblcustomerssilver


#  **Dividing New Records vs Old Records**

In [0]:
%sql
CREATE table DimCustomers
(
  DimCustomerKey BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id string,
  email string,
  city string,
  state string,
  domains string,
  full_name string,
  create_date date,
  update_date date
)

In [0]:
df_final.display()

## **SCD Type - 1**

In [0]:
from delta.tables import DeltaTable

In [0]:
%sql
Merge INTO DimCustomers
Using tblCustomersSilver
ON DimCustomers.customer_id = tblCustomersSilver.customer_id
WHEN MATCHED THEN UPDATE SET DimCustomers.domains  = tblCustomersSilver.domains, dimCustomers.update_date = current_timestamp()  -- changing attributes
WHEN NOT MATCHED THEN INSERT( customer_id, email, City, state, domains, full_name, create_date)
VALUES (tblCustomersSilver.customer_id, tblCustomersSilver.email, tblCustomersSilver.city, tblCustomersSilver.state, tblCustomersSilver.domains, tblCustomersSilver.full_name, current_timestamp())
;

In [0]:
%sql
SELECT * FROM dimcustomers